# ML-08 — Refresh / Content Opportunity Model

This notebook uses the Week 3 March→April data contract to compare a transparent rule, Logistic Regression, and Random Forest on the same held-out clients and ranking metrics. The June 2026 sample remains sealed.

## 1. Method choice and why

**Operational task:** rank visible pages for a content strategist’s next 20 review slots. A classifier’s probability is used as a ranking score; it is not an automatic edit decision.

**Primary model: Logistic Regression.** The proxy outcome is binary, there are only five contracted numeric features, and standardized coefficients are readable. **Challenger: Random Forest.** It can learn nonlinear interactions, but it is selected only if it improves validation precision@20 by at least 0.05 (one extra correct page in 20) without lowering average precision. This rule is fixed before the test set is read.

**Baseline:** the Week 4 baseline notebook is still an empty skeleton, so there is no honest prior numeric result to quote. Rather than compare against a different starter-data slice, I instantiate the transparent warehouse-compatible rule motivated by the earlier framing: prioritize pages with high March exposure, unstable/intermittent visibility, and a strong current position. The exact formula is declared below and never fitted to the outcome.

The label remains a future **proxy**—April average daily impressions at least 20% below March—not evidence that refreshing a page would cause recovery.

In [1]:
# Secure, self-contained setup and the same five-feature frame as Week 3.
import getpass
import importlib.util
import os
import re
import subprocess
import sys
import warnings

required = {
    "duckdb": "duckdb",
    "huggingface_hub": "huggingface_hub",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

warnings.filterwarnings("ignore", message="IProgress not found.*")
import duckdb
import numpy as np
import pandas as pd
import sklearn
from huggingface_hub import get_token
from IPython.display import display

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except (ImportError, KeyError, TypeError):
        pass
if not hf_token:
    hf_token = get_token()
if not hf_token:
    hf_token = getpass.getpass("Hugging Face READ token (input hidden): ")
if not (hf_token and hf_token.startswith("hf_") and len(hf_token) > 20 and not re.search(r"\s", hf_token)):
    raise RuntimeError("A valid Hugging Face READ token is required; do not paste it into a code cell.")

sql_safe_token = hf_token.replace("'", "''")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{sql_safe_token}')")
del hf_token, sql_safe_token

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"

feature_frame_sql = f"""
WITH march_page AS (
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS march_fact_days,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS march_impression_days,
        SUM(CASE
            WHEN gsc_impressions > 0 AND gsc_avg_position > 0
            THEN gsc_avg_position * gsc_impressions ELSE 0
        END) AS march_position_weighted_sum,
        SUM(CASE
            WHEN gsc_impressions > 0 AND gsc_avg_position > 0
            THEN gsc_impressions ELSE 0
        END) AS march_position_weight,
        STDDEV_SAMP(gsc_avg_position) FILTER (
            WHERE gsc_impressions > 0 AND gsc_avg_position > 0
        ) AS march_position_sd
    FROM {MARCH}
    GROUP BY 1, 2
), april_page AS (
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS april_fact_days,
        SUM(gsc_impressions) AS april_impressions
    FROM {APRIL}
    GROUP BY 1, 2
), eligible AS (
    SELECT
        m.*,
        a.april_fact_days,
        a.april_impressions,
        (a.april_impressions::DOUBLE / a.april_fact_days) /
        NULLIF(m.march_impressions::DOUBLE / m.march_fact_days, 0) - 1.0
            AS future_impression_change_pct
    FROM march_page AS m
    INNER JOIN april_page AS a USING (client_hash_id, content_hash_id)
    WHERE m.march_fact_days >= 20
      AND a.april_fact_days >= 20
      AND m.march_impressions >= 100
)
SELECT
    client_hash_id,
    content_hash_id,
    LN(1.0 + march_impressions) AS log_march_impressions,
    march_clicks::DOUBLE / NULLIF(march_impressions, 0) AS march_ctr,
    march_position_weighted_sum / NULLIF(march_position_weight, 0)
        AS march_weighted_position,
    march_impression_days::DOUBLE / march_fact_days AS march_impression_day_share,
    COALESCE(march_position_sd, 0.0) AS march_position_volatility,
    CASE WHEN future_impression_change_pct <= -0.20 THEN 1 ELSE 0 END
        AS future_visibility_decline
FROM eligible
ORDER BY client_hash_id, content_hash_id
"""

model_frame = con.sql(feature_frame_sql).df()
feature_cols = [
    "log_march_impressions",
    "march_ctr",
    "march_weighted_position",
    "march_impression_day_share",
    "march_position_volatility",
]
label_col = "future_visibility_decline"
context_cols = ["client_hash_id", "content_hash_id"]

assert len(model_frame) == 100_049
assert len(feature_cols) == 5
assert not model_frame.duplicated(context_cols).any()
assert model_frame[label_col].isin([0, 1]).all()
assert all("april" not in feature.lower() and "future" not in feature.lower() for feature in feature_cols)

feature_missingness = model_frame[feature_cols].isna().mean().rename("missing_share").reset_index()
feature_missingness.columns = ["feature", "missing_share"]
print(f"Model frame: {len(model_frame):,} page-decisions | {model_frame['client_hash_id'].nunique()} clients")
print(f"Overall future-decline proxy rate: {model_frame[label_col].mean():.1%}")
print(f"scikit-learn version: {sklearn.__version__} | random seeds: 42 and 43")
display(feature_missingness)

Model frame: 100,049 page-decisions | 41 clients
Overall future-decline proxy rate: 50.5%
scikit-learn version: 1.9.0 | random seeds: 42 and 43


,feature,missing_share
0,log_march_impressions,0.0
1,march_ctr,0.0
2,march_weighted_position,0.0
3,march_impression_day_share,0.0
4,march_position_volatility,0.0


## 2. Split design

The time order is fixed first: March features predict the separate April proxy outcome. I then split by `client_hash_id`, not by page, so no client contributes pages to more than one set.

- **Training clients:** fit candidate models.
- **Validation clients:** choose Logistic Regression versus Random Forest using the predeclared complexity gate.
- **Test clients:** touched once for the final model-vs-baseline table.

The first grouped split reserves roughly 20% of clients for test (`random_state=42`); the second reserves roughly 20% of all clients for validation (`random_state=43`). Group sizes are uneven, so row shares and outcome rates need not be equal. That is a feature of honest client transfer, not something to repair with a page-random split.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

outer_split = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
development_idx, test_idx = next(
    outer_split.split(model_frame, model_frame[label_col], model_frame["client_hash_id"])
)
development = model_frame.iloc[development_idx].copy()
test = model_frame.iloc[test_idx].copy()

inner_split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=43)
train_rel_idx, validation_rel_idx = next(
    inner_split.split(development, development[label_col], development["client_hash_id"])
)
train = development.iloc[train_rel_idx].copy()
validation = development.iloc[validation_rel_idx].copy()

train_clients = set(train["client_hash_id"])
validation_clients = set(validation["client_hash_id"])
test_clients = set(test["client_hash_id"])
assert train_clients.isdisjoint(validation_clients)
assert train_clients.isdisjoint(test_clients)
assert validation_clients.isdisjoint(test_clients)
assert len(train) + len(validation) + len(test) == len(model_frame)

split_summary = pd.DataFrame(
    {
        "split": ["Train", "Validation", "Test"],
        "rows": [len(train), len(validation), len(test)],
        "clients": [len(train_clients), len(validation_clients), len(test_clients)],
        "future_decline_rate": [
            train[label_col].mean(),
            validation[label_col].mean(),
            test[label_col].mean(),
        ],
    }
)
split_summary["future_decline_rate"] = split_summary["future_decline_rate"].round(3)
display(split_summary)
print("PASS: feature/outcome time windows do not overlap, and client groups are disjoint.")

,split,rows,clients,future_decline_rate
0,Train,50639,24,0.389
1,Validation,35582,8,0.594
2,Test,13828,9,0.704


PASS: feature/outcome time windows do not overlap, and client groups are disjoint.


## 3. Train + compare vs my baseline

**Same rows, split, and metrics:** every final method scores the same 13,828 held-out test page-decisions. The primary metric is precision@20; precision@100 shows whether the result survives deeper in the queue, while average precision and ROC-AUC summarize the full ranking.

**Transparent baseline formula (no label fitting):**

`log(1 + March impressions) × (1 + position volatility) × (2 − impression-day share) ÷ (1 + weighted position)`

This favors high-exposure pages whose visibility appears unstable or intermittent, with stronger current positions receiving more review priority. A missing position is filled with the development-set median—never a test-set statistic.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def precision_at_k(scores, labels, k):
    scores = np.asarray(scores)
    labels = np.asarray(labels)
    if len(scores) != len(labels) or len(scores) < k:
        raise ValueError("scores and labels must have equal length and at least k rows")
    top_idx = np.argsort(-scores, kind="stable")[:k]
    return float(labels[top_idx].mean())

def ranking_metrics(scores, labels):
    return {
        "precision_at_20": precision_at_k(scores, labels, 20),
        "precision_at_100": precision_at_k(scores, labels, 100),
        "average_precision": average_precision_score(labels, scores),
        "roc_auc": roc_auc_score(labels, scores),
    }

def make_logistic():
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
        ]
    )

def make_forest():
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=300,
                max_depth=10,
                min_samples_leaf=20,
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=42,
            )),
        ]
    )

# Model choice happens on validation clients only.
logistic_validation = make_logistic().fit(train[feature_cols], train[label_col])
forest_validation = make_forest().fit(train[feature_cols], train[label_col])
validation_scores = {
    "Logistic Regression": logistic_validation.predict_proba(validation[feature_cols])[:, 1],
    "Random Forest": forest_validation.predict_proba(validation[feature_cols])[:, 1],
}
validation_rows = []
for method, scores in validation_scores.items():
    validation_rows.append({"method": method, **ranking_metrics(scores, validation[label_col])})
validation_table = pd.DataFrame(validation_rows)

lr_validation = validation_table.set_index("method").loc["Logistic Regression"]
rf_validation = validation_table.set_index("method").loc["Random Forest"]
forest_earns_complexity = (
    rf_validation["precision_at_20"] >= lr_validation["precision_at_20"] + 0.05
    and rf_validation["average_precision"] >= lr_validation["average_precision"]
)
selected_name = "Random Forest" if forest_earns_complexity else "Logistic Regression"

display(validation_table.round(3))
print(f"Selected before test evaluation: {selected_name}")
print("Complexity gate:", "PASSED" if forest_earns_complexity else "NOT PASSED")

,method,precision_at_20,precision_at_100,average_precision,roc_auc
0,Logistic Regression,1.00,0.82,0.702,0.653
1,Random Forest,0.85,0.80,0.711,0.661


Selected before test evaluation: Logistic Regression
Complexity gate: NOT PASSED


In [4]:
# Refit fixed candidates on development clients, then touch test once.
logistic_final = make_logistic().fit(development[feature_cols], development[label_col])
forest_final = make_forest().fit(development[feature_cols], development[label_col])

development_position_median = development["march_weighted_position"].median()
baseline_position = test["march_weighted_position"].fillna(development_position_median)
baseline_scores = (
    test["log_march_impressions"]
    * (1.0 + test["march_position_volatility"])
    * (2.0 - test["march_impression_day_share"])
    / (1.0 + baseline_position)
)

test_scores = {
    "Transparent baseline rule": baseline_scores.to_numpy(),
    "Logistic Regression": logistic_final.predict_proba(test[feature_cols])[:, 1],
    "Random Forest": forest_final.predict_proba(test[feature_cols])[:, 1],
}
test_base_rate = float(test[label_col].mean())
comparison_rows = [
    {
        "method": "Random/base-rate reference",
        "precision_at_20": test_base_rate,
        "precision_at_100": test_base_rate,
        "average_precision": test_base_rate,
        "roc_auc": 0.5,
        "test_rows": len(test),
    }
]
for method, scores in test_scores.items():
    comparison_rows.append(
        {"method": method, **ranking_metrics(scores, test[label_col]), "test_rows": len(test)}
    )
comparison_table = pd.DataFrame(comparison_rows)
comparison_table["selected_on_validation"] = comparison_table["method"].eq(selected_name)
metric_cols = ["precision_at_20", "precision_at_100", "average_precision", "roc_auc"]
comparison_table[metric_cols] = comparison_table[metric_cols].round(3)

assert comparison_table["test_rows"].nunique() == 1
assert comparison_table["test_rows"].iloc[0] == len(test)
display(comparison_table)

selected_test_scores = test_scores[selected_name]
selected_test_model = logistic_final if selected_name == "Logistic Regression" else forest_final
selected_p20 = comparison_table.loc[
    comparison_table["method"].eq(selected_name), "precision_at_20"
].iloc[0]
baseline_p20 = comparison_table.loc[
    comparison_table["method"].eq("Transparent baseline rule"), "precision_at_20"
].iloc[0]
print(f"Primary result: {selected_name} precision@20 = {selected_p20:.3f}; baseline = {baseline_p20:.3f}")

,method,precision_at_20,precision_at_100,average_precision,roc_auc,test_rows,selected_on_validation
0,Random/base-rate reference,0.704,0.704,0.704,0.500,13828,False
1,Transparent baseline rule,0.800,0.660,0.720,0.525,13828,False
2,Logistic Regression,0.800,0.850,0.801,0.658,13828,True
3,Random Forest,0.900,0.860,0.779,0.626,13828,False


Primary result: Logistic Regression precision@20 = 0.800; baseline = 0.800


## 4. Errors and interpretation

I interpret the model selected on validation clients, not whichever method happens to look best on test. Permutation importance measures the average-precision drop when one test feature is shuffled; standardized Logistic Regression coefficients show direction. Both are associations with the proxy outcome, not causal search-ranking factors.

For error reading, the 0.50 probability threshold is diagnostic only—the product decision remains the top-20 ranking. Concrete cases below are stripped of content/client IDs and show only safe aggregate features.

In [5]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import confusion_matrix, precision_score, recall_score

permutation = permutation_importance(
    selected_test_model,
    test[feature_cols],
    test[label_col],
    scoring="average_precision",
    n_repeats=5,
    random_state=42,
    n_jobs=-1,
    max_samples=5_000,
)
importance_table = pd.DataFrame(
    {
        "feature": feature_cols,
        "permutation_AP_drop": permutation.importances_mean,
        "permutation_std": permutation.importances_std,
    }
)
if selected_name == "Logistic Regression":
    importance_table["standardized_coefficient"] = selected_test_model.named_steps["model"].coef_[0]
else:
    importance_table["native_importance"] = selected_test_model.named_steps["model"].feature_importances_
importance_table = importance_table.sort_values("permutation_AP_drop", ascending=False).reset_index(drop=True)
importance_table["top_three"] = importance_table.index < 3
display(importance_table.round(3))

threshold_predictions = (selected_test_scores >= 0.50).astype(int)
tn, fp, fn, tp = confusion_matrix(test[label_col], threshold_predictions).ravel()
rank_order = np.argsort(-selected_test_scores, kind="stable")
top20_rows = test.iloc[rank_order[:20]]
threshold_summary = pd.DataFrame(
    {
        "diagnostic": [
            "True negatives at 0.50",
            "False positives at 0.50",
            "False negatives at 0.50",
            "True positives at 0.50",
            "Threshold precision",
            "Threshold recall",
            "False positives inside top 20",
        ],
        "value": [
            tn,
            fp,
            fn,
            tp,
            precision_score(test[label_col], threshold_predictions),
            recall_score(test[label_col], threshold_predictions),
            int((top20_rows[label_col] == 0).sum()),
        ],
    }
)
display(threshold_summary.round(3))

# Three public-safe wrong cases: two top-20 false alarms and one strongly missed decline.
error_frame = test[feature_cols + [label_col]].copy()
error_frame["model_score"] = selected_test_scores
error_frame["threshold_prediction"] = threshold_predictions
ranked_errors = error_frame.iloc[rank_order]
top20_false_positives = ranked_errors.head(20).loc[ranked_errors.head(20)[label_col].eq(0)].head(2).copy()
strong_false_negative = error_frame.loc[
    error_frame[label_col].eq(1) & error_frame["threshold_prediction"].eq(0)
].nsmallest(1, "model_score").copy()
wrong_cases = pd.concat([top20_false_positives, strong_false_negative], ignore_index=True)
wrong_cases["case"] = ["FP-1", "FP-2", "FN-1"]
wrong_cases["error_type"] = ["Top-20 false positive", "Top-20 false positive", "False negative at 0.50"]
wrong_cases["approx_march_impressions"] = np.expm1(wrong_cases["log_march_impressions"]).round().astype(int)
wrong_cases["march_ctr_pct"] = (100 * wrong_cases["march_ctr"]).round(2)
wrong_cases["why_hard"] = [
    "Zero CTR and strong position looked vulnerable, but April did not cross the decline proxy.",
    "Zero CTR and strong position looked vulnerable, but April did not cross the decline proxy.",
    "Healthy March CTR/position looked safe, yet April still crossed the decline proxy.",
]
case_view = wrong_cases[
    [
        "case",
        "error_type",
        "approx_march_impressions",
        "march_ctr_pct",
        "march_weighted_position",
        "march_impression_day_share",
        "march_position_volatility",
        "model_score",
        label_col,
        "why_hard",
    ]
].copy()
numeric_case_cols = [
    "march_weighted_position",
    "march_impression_day_share",
    "march_position_volatility",
    "model_score",
]
case_view[numeric_case_cols] = case_view[numeric_case_cols].round(3)
display(case_view)

,feature,permutation_AP_drop,permutation_std,standardized_coefficient,top_three
0,march_impression_day_share,0.046,0.001,0.311,True
1,march_ctr,0.037,0.004,-0.431,True
2,march_weighted_position,0.022,0.003,-0.168,True
3,log_march_impressions,0.013,0.002,-0.187,False
4,march_position_volatility,0.003,0.001,0.039,False


,diagnostic,value
0,True negatives at 0.50,2492.000
1,False positives at 0.50,1597.000
2,False negatives at 0.50,3694.000
3,True positives at 0.50,6045.000
4,Threshold precision,0.791
5,Threshold recall,0.621
6,False positives inside top 20,4.000


,case,error_type,approx_march_impressions,march_ctr_pct,march_weighted_position,march_impression_day_share,march_position_volatility,model_score,future_visibility_decline,why_hard
0,FP-1,Top-20 false positive,118,0.00,4.564,1.0,14.928,0.697,0,Zero CTR and strong position looked vulnerable...
1,FP-2,Top-20 false positive,102,0.00,3.302,1.0,4.069,0.691,0,Zero CTR and strong position looked vulnerable...
2,FN-1,False negative at 0.50,286,9.44,7.518,1.0,4.591,0.000,1,"Healthy March CTR/position looked safe, yet Ap..."


### What the results say

The validation decision selected **Logistic Regression**: its validation precision@20 was 1.00 versus 0.85 for Random Forest, so the nonlinear challenger did not earn its complexity. On the untouched test clients, Logistic Regression and the transparent baseline both reached **precision@20 = 0.80** (16 of 20). Logistic Regression was stronger deeper in the ranking—precision@100 = 0.85, average precision = 0.801, ROC-AUC = 0.658—but it failed the earlier requirement of beating the baseline by 0.10 at the actual 20-slot decision point. Random Forest reached 0.90 on test precision@20, but promoting it after its weaker validation result would reward one favorable split.

The top three permutation signals were March CTR, impression-day share, and weighted position. Lower CTR and stronger/persistent March visibility received more decline risk from the linear model; that may reflect greater downside exposure or client mix, not causation. Position volatility contributed very little under permutation, which challenges the baseline’s instability assumption.

The errors show the timing limit of a one-month snapshot: top-ranked false positives often had zero March CTR and strong positions but did not decline in April, while a strongly missed positive looked healthy in March and declined anyway. At the 0.50 diagnostic threshold the selected model had precision about 0.79 and recall about 0.62, leaving many false negatives. The proxy rate also shifted from 38.9% in training clients to 70.4% in test clients, so client transfer and calibration—not extra model complexity—are the next problems to solve.

**Decision:** keep the transparent rule as the operational default for now. Logistic Regression is useful directional evidence and a stronger full-ranking research candidate, but it has not yet earned replacement at precision@20.

## 5. Self-check

- [x] Method choice fits a probability-ranked binary proxy and starts simple.
- [x] March features and April outcome are separated; June remains sealed.
- [x] Train, validation, and test clients are disjoint.
- [x] Baseline and models use the same test rows and metrics.
- [x] The comparison includes base rate, precision@20/100, average precision, and ROC-AUC.
- [x] Model choice was made on validation, not test.
- [x] Top features and three public-safe wrong cases are interpreted.
- [x] Complexity is not rewarded for one favorable test result.
- [x] No client names, content IDs, URLs, private queries, future features, or token values are displayed.
- [x] Claims are observed, directional decision-support—not causal.

In [6]:
# Final machine-check, followed by the non-negotiable same-split comparison table.
assert selected_name == "Logistic Regression"
assert not forest_earns_complexity
assert train_clients.isdisjoint(validation_clients | test_clients)
assert validation_clients.isdisjoint(test_clients)
assert comparison_table["test_rows"].nunique() == 1
assert len(feature_cols) == 5
assert {"client_hash_id", "content_hash_id", label_col}.isdisjoint(feature_cols)
assert all("future" not in feature and "april" not in feature for feature in feature_cols)
assert len(case_view) == 3
assert comparison_table["method"].eq("Transparent baseline rule").sum() == 1

print("SELF-CHECK PASS: grouped selection | same-split baseline comparison | errors read | no leakage")
display(comparison_table)

SELF-CHECK PASS: grouped selection | same-split baseline comparison | errors read | no leakage


,method,precision_at_20,precision_at_100,average_precision,roc_auc,test_rows,selected_on_validation
0,Random/base-rate reference,0.704,0.704,0.704,0.500,13828,False
1,Transparent baseline rule,0.800,0.660,0.720,0.525,13828,False
2,Logistic Regression,0.800,0.850,0.801,0.658,13828,True
3,Random Forest,0.900,0.860,0.779,0.626,13828,False
